# Temporal Extensions — Three Priority Experiments

This notebook implements three experiments that extend the core `temporal_analysis.ipynb` pipeline:

| # | Extension | Question answered |
|---|---|---|
| 1 | **KLS AUC Evaluation** | Do multi-week symbolic patterns (KarmaLego 2-TIRPs) beat the 15-feature baseline? |
| 2 | **Earnings-Week Flag** | Does a binary earnings-season flag outperform cyclic calendar encodings? |
| 3 | **Temporal + Baseline Combo** | Do the 7 cyclic calendar features lift AUC when stacked on top of the 15 baseline features? |

**Walk-forward setup:** 6 expanding-window folds (test years 2019–2024). The 2020 COVID fold is evaluated but excluded from mean AUC.  
**Models:** L2 LogisticRegression (StandardScaler pipeline) + XGBoost (scale-invariant, tree-based).

---
### How to use this notebook
Run cells top-to-bottom once to define all helpers and functions.  
Each Extension section is then self-contained — re-run any single extension without restarting the kernel.

## 1  Imports & Configuration

Standard scientific stack plus scikit-learn pipelines and XGBoost.  
All warnings are suppressed to keep output clean.

In [ ]:
from __future__ import annotations

import sys, warnings, time
warnings.filterwarnings('ignore')

from bisect import bisect_left, bisect_right
from collections import defaultdict
from dataclasses import dataclass
from enum import IntEnum
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score

try:
    from xgboost import XGBClassifier
    _HAS_XGB = True
    print('XGBoost available:', XGBClassifier.__module__)
except ImportError:
    _HAS_XGB = False
    print('[warn] xgboost not installed — XGBoost cells will be skipped.')


## 2  Paths, Feature Lists & Hyperparameters

- **`BASELINE_FEATURES`** — 15 technical + fundamental indicators used in the main model.
- **`TEMPORAL_FEATURES`** — 7 cyclic calendar encodings already present in `dataset_clean_temporal.csv`.
- **`KLS_FEATURES`** — 5-feature subset used for KarmaLego (keeps KARMA O(n²) step fast).
- **KarmaLego params** — `MAX_K=2` (2-TIRPs only) with early-exit inner-loop optimisation.

In [ ]:
PROJECT_ROOT  = Path('..').resolve()
DATASET_PATH  = PROJECT_ROOT / 'structured_csv_data_files' / 'fetched_data' / 'dataset_clean.csv'
TEMPORAL_PATH = PROJECT_ROOT / 'structured_csv_data_files' / 'fetched_data' / 'dataset_clean_temporal.csv'
TARGET        = 'GapUp'

BASELINE_FEATURES = [
    'MACD', 'ROC', 'StochPercK',
    'CloseVEma50', 'CloseVSma20', 'ADX',
    'BollingerBandWidth', 'ATR', 'FiveDStdDev',
    'OBV', 'MFI', 'VolumeRatio',
    'NetMargin', 'RoA', 'RevGrowthQoQ',
]

TEMPORAL_FEATURES = [
    'Month_sin', 'Month_cos',
    'WeekOfYear_sin', 'WeekOfYear_cos',
    'Quarter_sin', 'Quarter_cos',
    'DaysSinceStart',
]

# 5 momentum/breadth features for KarmaLego — keeps interval count small
KLS_FEATURES = ['MACD', 'ROC', 'StochPercK', 'BollingerBandWidth', 'VolumeRatio']
STATE_LABELS  = ['low', 'mid', 'high']

# Walk-forward folds (expanding window)
FOLDS = [
    {'train_years': list(range(2016, 2019)), 'test_year': 2019},
    {'train_years': list(range(2016, 2020)), 'test_year': 2020},
    {'train_years': list(range(2016, 2021)), 'test_year': 2021},
    {'train_years': list(range(2016, 2022)), 'test_year': 2022},
    {'train_years': list(range(2016, 2023)), 'test_year': 2023},
    {'train_years': list(range(2016, 2024)), 'test_year': 2024},
]

# KarmaLego hyperparameters
MIN_TICKER_SUPPORT = 8   # minimum number of tickers a TIRP must appear in
MAX_GAP            = 8   # max gap (weeks) allowed between two intervals
MAX_K              = 2   # only mine 2-TIRPs (avoids expensive LEGO expansion)
MAX_LAG            = 4   # max weeks after TIRP end to flag as KLS feature

print(f'Baseline features : {len(BASELINE_FEATURES)}')
print(f'Temporal features : {len(TEMPORAL_FEATURES)}')
print(f'KLS features      : {len(KLS_FEATURES)}')
print(f'Walk-forward folds: {len(FOLDS)}')


## 3  Shared Evaluation Helpers

Three reusable functions used by all three extensions:

- **`load_primary(path)`** — loads any project CSV, drops the COVID distortion window (`is_extreme_event == 1`), and adds a per-ticker `WeekIdx` column.
- **`walk_forward(X, y, years, make_clf, label)`** — expanding-window walk-forward evaluation; returns a DataFrame with AUC and accuracy per fold.
- **`print_results(df, title)`** — pretty-prints the fold table plus non-COVID mean AUC/accuracy.

In [ ]:
def load_primary(path: Path) -> pd.DataFrame:
    """Load CSV, parse dates, filter out COVID extreme-event rows."""
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Date'], utc=True).dt.tz_convert(None)
    primary = df[df['is_extreme_event'] == 0].copy()
    primary = primary.sort_values(['Ticker', 'Date']).reset_index(drop=True)
    primary['WeekIdx'] = primary.groupby('Ticker').cumcount()
    return primary


def make_logreg() -> Pipeline:
    """L2 LogisticRegression inside a StandardScaler pipeline."""
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            penalty='l2', C=1.0, solver='liblinear',
            max_iter=1000, random_state=42
        )),
    ])


def make_xgb():
    """Conservative XGBoost — scale-invariant, low learning rate."""
    return XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        eval_metric='auc', tree_method='hist', random_state=42, n_jobs=-1,
    )


def walk_forward(
    X_all: np.ndarray, y_all: np.ndarray,
    year_all: np.ndarray, make_clf, label: str
) -> pd.DataFrame:
    """Expanding-window walk-forward evaluation over FOLDS."""
    records = []
    for fold in FOLDS:
        tr = np.isin(year_all, fold['train_years'])
        te = year_all == fold['test_year']
        X_tr, X_te = X_all[tr], X_all[te]
        y_tr, y_te = y_all[tr], y_all[te]
        if len(y_te) == 0 or len(np.unique(y_te)) < 2:
            continue
        clf = make_clf()
        clf.fit(X_tr, y_tr)
        proba = clf.predict_proba(X_te)[:, 1]
        pred  = (proba >= 0.5).astype(int)
        records.append({
            'Label':      label,
            'Test year':  fold['test_year'],
            'AUC':        round(roc_auc_score(y_te, proba), 4),
            'Accuracy':   round(accuracy_score(y_te, pred), 4),
            'COVID':      fold['test_year'] == 2020,
            'Train rows': int(tr.sum()),
            'Test rows':  int(te.sum()),
        })
    return pd.DataFrame(records)


def print_results(df: pd.DataFrame, title: str) -> None:
    nc = df[~df['COVID']]
    print(f"\n{'='*62}")
    print(f'  {title}')
    print(f"{'='*62}")
    print(df.to_string(index=False))
    print(f"\n  Mean AUC  (non-COVID): {nc['AUC'].mean():.4f}")
    print(f"  Mean Acc  (non-COVID): {nc['Accuracy'].mean():.4f}")

print('Helpers defined.')


## 4  TD4C Supervised Discretisation

**TD4C** (Temporal Discretisation for Classification) finds the cutpoints that maximise information gain between the discretised feature and the binary target (`GapUp`).  
We use **3 states** (`low / mid / high`) and **19 candidate percentile thresholds** (5th–95th percentile, evenly spaced).

The cutpoints are fit once on the 2016–2018 training window and then applied to the full history to avoid look-ahead bias.

In [ ]:
def _binary_entropy(p: float) -> float:
    if p <= 0 or p >= 1:
        return 0.0
    return -p * np.log2(p) - (1.0 - p) * np.log2(1.0 - p)


def td4c_cutpoints(
    values: np.ndarray, labels: np.ndarray,
    n_states: int = 3, n_candidates: int = 19
) -> list[float]:
    """Return n_states-1 supervised cutpoints via information-gain search."""
    pcts       = np.linspace(5, 95, n_candidates)
    candidates = np.unique(np.percentile(values, pcts))
    if len(candidates) < n_states - 1:
        return [float(np.median(values))] * (n_states - 1)
    N     = len(labels)
    H_Y   = _binary_entropy(labels.mean())
    best_ig, best_cuts = -np.inf, []
    if n_states == 2:
        for c in candidates:
            lo = values < c
            n_l, n_h = int(lo.sum()), N - int(lo.sum())
            if n_l == 0 or n_h == 0:
                continue
            H_c = (n_l/N)*_binary_entropy(labels[lo].mean()) + \
                  (n_h/N)*_binary_entropy(labels[~lo].mean())
            if (ig := H_Y - H_c) > best_ig:
                best_ig, best_cuts = ig, [float(c)]
    else:
        for i, c1 in enumerate(candidates):
            for c2 in candidates[i+1:]:
                m_lo = values < c1
                m_md = (values >= c1) & (values < c2)
                m_hi = values >= c2
                n_lo, n_md, n_hi = m_lo.sum(), m_md.sum(), m_hi.sum()
                if n_lo == 0 or n_md == 0 or n_hi == 0:
                    continue
                H_c = ((n_lo/N)*_binary_entropy(labels[m_lo].mean()) +
                       (n_md/N)*_binary_entropy(labels[m_md].mean()) +
                       (n_hi/N)*_binary_entropy(labels[m_hi].mean()))
                if (ig := H_Y - H_c) > best_ig:
                    best_ig, best_cuts = ig, [float(c1), float(c2)]
    return best_cuts


def apply_cutpoints(
    values: np.ndarray, cuts: list[float], labels: list[str]
) -> np.ndarray:
    """Map numeric values to state labels using pre-fit cutpoints."""
    idx = np.searchsorted(cuts, values, side='right')
    return np.asarray(labels, dtype=object)[idx]

print('TD4C functions defined.')


## 5  Symbolic Interval Representation

Each continuous time series is converted to a sequence of **symbolic intervals** — maximal contiguous runs of the same discretised state (e.g. `MACD=high` from week 12 to week 15).  

The `Interval` dataclass stores `(start, end, variable, state)`.  Its `.symbol` property produces the string `variable=state` used as the KarmaLego alphabet.

**Allen's 7 temporal relations** (`before`, `meets`, `overlaps`, `finished-by`, `contains`, `starts`, `equal`) describe how two intervals relate in time.  
The `allen_relation()` function returns `None` when two intervals are more than `max_gap` weeks apart — this is the pruning that makes KarmaLego tractable.

In [ ]:
@dataclass(frozen=True, order=True)
class Interval:
    start:    int
    end:      int
    variable: str
    state:    str

    @property
    def symbol(self) -> str:
        return f'{self.variable}={self.state}'

    @property
    def duration(self) -> int:
        return self.end - self.start + 1


def _merge_symbols(
    variable: str, symbols: np.ndarray, week_idx: np.ndarray
) -> list[Interval]:
    """Merge consecutive identical symbols into Interval objects."""
    if len(symbols) == 0:
        return []
    out, cur_sym = [], symbols[0]
    cur_start = cur_end = int(week_idx[0])
    for k in range(1, len(symbols)):
        if symbols[k] == cur_sym and week_idx[k] == cur_end + 1:
            cur_end = int(week_idx[k])
        else:
            out.append(Interval(cur_start, cur_end, variable, str(cur_sym)))
            cur_sym   = symbols[k]
            cur_start = cur_end = int(week_idx[k])
    out.append(Interval(cur_start, cur_end, variable, str(cur_sym)))
    return out


def build_ticker_intervals(
    ticker_df: pd.DataFrame,
    cuts: dict[str, list[float]],
    features: list[str]
) -> list[Interval]:
    """Convert one ticker's numeric time-series to sorted symbolic intervals."""
    week_idx = ticker_df['WeekIdx'].to_numpy()
    out: list[Interval] = []
    for feat in features:
        syms = apply_cutpoints(
            ticker_df[feat].to_numpy(dtype=float), cuts[feat], STATE_LABELS
        )
        out.extend(_merge_symbols(feat, syms, week_idx))
    out.sort()
    return out


class AllenRelation(IntEnum):
    BEFORE      = 0
    MEETS       = 1
    OVERLAPS    = 2
    FINISHED_BY = 3
    CONTAINS    = 4
    STARTS      = 5
    EQUAL       = 6


def allen_relation(a: Interval, b: Interval, max_gap: int):
    """Return Allen relation a→b, or None if gap exceeds max_gap."""
    if a.start == b.start:
        return AllenRelation.EQUAL if a.end == b.end else AllenRelation.STARTS
    if a.end + 1 < b.start:
        return None if (b.start - a.end - 1) > max_gap else AllenRelation.BEFORE
    if a.end + 1 == b.start:
        return AllenRelation.MEETS
    if a.end < b.end:
        return AllenRelation.OVERLAPS
    if a.end == b.end:
        return AllenRelation.FINISHED_BY
    return AllenRelation.CONTAINS

print('Interval + Allen relation helpers defined.')


## 6  KarmaLego TIRP Mining

**KarmaLego** mines frequent **Temporal Interval Relationship Patterns (TIRPs)** across multiple entities (tickers).

- **KARMA** step — enumerate all valid ordered interval pairs per ticker and count how many tickers each `(symbol_A, symbol_B, Allen_relation)` triple appears in.  Keep pairs with `vertical_support >= MIN_TICKER_SUPPORT`.
- **LEGO** step — extend frequent k-TIRPs to (k+1)-TIRPs by appending one more interval.  Here `MAX_K=2` so we stop after 2-TIRPs.

**Key optimisation:** intervals are sorted by `start` time.  The inner KARMA loop breaks as soon as `b.start > a.end + max_gap + 1`, turning worst-case O(n²) into roughly O(n × w) where w is the average window of co-occurring intervals.  This cut runtime from >17 minutes to ~75 seconds.

**KLS feature matrix** — for each mined TIRP, `build_kls_features()` creates a binary column: `1` if that TIRP was realised within `MAX_LAG` weeks before the current observation, `0` otherwise.

In [ ]:
@dataclass
class TIRP:
    symbols:             tuple[str, ...]
    relations:           list[list[AllenRelation]]
    instances_by_ticker: dict[str, list[tuple[int, ...]]]

    @property
    def k(self) -> int:
        return len(self.symbols)

    @property
    def vertical_support(self) -> int:
        return len(self.instances_by_ticker)

    def rel(self, i: int, j: int) -> AllenRelation:
        return self.relations[i][j - i - 1]


def karmalego_multi(
    ticker_intervals_mining: dict[str, list[Interval]],
    min_ticker_support: int, max_gap: int,
    max_k: int = 2, verbose: bool = True
) -> list[TIRP]:
    """Mine frequent TIRPs by vertical (ticker) support."""
    per_ticker_by_symbol: dict[str, dict[str, list[int]]] = {}
    for t, ivls in ticker_intervals_mining.items():
        bs: dict[str, list[int]] = defaultdict(list)
        for idx, iv in enumerate(ivls):
            bs[iv.symbol].append(idx)
        per_ticker_by_symbol[t] = bs

    # KARMA — frequent 2-TIRPs with early-exit optimisation
    pair_instances: dict = defaultdict(dict)
    for t, ivls in ticker_intervals_mining.items():
        n = len(ivls)
        for i in range(n):
            a      = ivls[i]
            cutoff = a.end + max_gap + 1   # break when b.start exceeds this
            for j in range(i + 1, n):
                b = ivls[j]
                if b.start > cutoff:
                    break                  # sorted by start → safe to stop
                r = allen_relation(a, b, max_gap)
                if r is None:
                    continue
                pair_instances[(a.symbol, b.symbol, r)].setdefault(t, []).append((i, j))

    frequent_pairs = {k: d for k, d in pair_instances.items()
                      if len(d) >= min_ticker_support}
    if verbose:
        print(f'  KARMA: {len(frequent_pairs):,} frequent 2-TIRPs '
              f'(vert_support >= {min_ticker_support})')

    tirps_k = [
        TIRP(symbols=(s0, s1), relations=[[r]],
             instances_by_ticker={t: list(insts) for t, insts in by_t.items()})
        for (s0, s1, r), by_t in frequent_pairs.items()
    ]
    all_tirps     = list(tirps_k)
    current_level = tirps_k

    if max_k < 3:
        return all_tirps

    # LEGO — extend to k >= 3 (not used in this notebook, MAX_K=2)
    last_to_candidates: dict[str, list] = defaultdict(list)
    for (s_prev, s_new, r) in frequent_pairs:
        last_to_candidates[s_prev].append((s_new, r))

    for target_k in range(3, max_k + 1):
        next_level: list[TIRP] = []
        for tirp in current_level:
            last_sym = tirp.symbols[-1]
            for (s_new, r_last_new) in last_to_candidates.get(last_sym, []):
                groups: dict = defaultdict(lambda: defaultdict(list))
                for t, t_insts in tirp.instances_by_ticker.items():
                    ivls   = ticker_intervals_mining[t]
                    bs     = per_ticker_by_symbol[t]
                    for inst in t_insts:
                        last_idx = inst[-1]
                        last_iv  = ivls[last_idx]
                        cutoff   = last_iv.end + max_gap + 1
                        for j in bs.get(s_new, []):
                            if j <= last_idx:
                                continue
                            if ivls[j].start > cutoff:
                                break
                            r_last = allen_relation(last_iv, ivls[j], max_gap)
                            if r_last != r_last_new:
                                continue
                            rels_to_j, bad = [], False
                            for m in range(len(inst) - 1):
                                r_m = allen_relation(ivls[inst[m]], ivls[j], max_gap)
                                if r_m is None:
                                    bad = True; break
                                rels_to_j.append(r_m)
                            if bad:
                                continue
                            rels_to_j.append(r_last_new)
                            groups[tuple(rels_to_j)][t].append(inst + (j,))
                for key, per_t in groups.items():
                    if len(per_t) < min_ticker_support:
                        continue
                    new_rels = [row[:] for row in tirp.relations]
                    for i in range(len(new_rels)):
                        new_rels[i].append(key[i])
                    new_rels.append([key[-1]])
                    next_level.append(TIRP(
                        symbols=tirp.symbols + (s_new,),
                        relations=new_rels,
                        instances_by_ticker={t: sorted(set(v)) for t, v in per_t.items()},
                    ))
        if verbose:
            print(f'  LEGO k={target_k}: {len(next_level):,} frequent TIRPs')
        if not next_level:
            break
        all_tirps.extend(next_level)
        current_level = next_level

    return all_tirps


def instantiate_tirp(
    tirp: TIRP, ivls: list[Interval], max_gap: int
) -> list[tuple[int, ...]]:
    """Find all realisations of `tirp` in a single ticker's interval list."""
    by_symbol: dict[str, list[int]] = defaultdict(list)
    for idx, iv in enumerate(ivls):
        by_symbol[iv.symbol].append(idx)
    k, target_syms = tirp.k, tirp.symbols
    results: list[tuple[int, ...]] = []

    def backtrack(partial: tuple[int, ...]):
        pos = len(partial)
        if pos == k:
            results.append(partial); return
        for idx in by_symbol.get(target_syms[pos], []):
            if partial and idx <= partial[-1]:
                continue
            if all(allen_relation(ivls[partial[m]], ivls[idx], max_gap) == tirp.rel(m, pos)
                   for m in range(pos)):
                backtrack(partial + (idx,))

    backtrack(())
    return results


def build_kls_features(
    tirps: list[TIRP],
    ticker_intervals: dict[str, list[Interval]],
    anchors: pd.DataFrame,
    max_gap: int, max_lag: int
) -> tuple[np.ndarray, list[str]]:
    """Binary (n_anchors x n_tirps) KLS feature matrix."""
    N, P = len(anchors), len(tirps)
    anchor_by_ticker: dict[str, tuple[np.ndarray, np.ndarray]] = {}
    for t, sub in anchors.groupby('Ticker'):
        sub = sub.sort_values('WeekIdx')
        anchor_by_ticker[t] = (sub['WeekIdx'].to_numpy(), sub.index.to_numpy())

    X = np.zeros((N, P), dtype=np.int8)
    for p, tirp in enumerate(tirps):
        for t, ivls in ticker_intervals.items():
            if t not in anchor_by_ticker:
                continue
            insts = instantiate_tirp(tirp, ivls, max_gap)
            if not insts:
                continue
            weeks_sorted, rows_sorted = anchor_by_ticker[t]
            for inst in insts:
                e  = max(ivls[idx].end for idx in inst)
                lo = bisect_left(weeks_sorted, e + 1)
                hi = bisect_right(weeks_sorted, e + max_lag)
                if hi > lo:
                    X[rows_sorted[lo:hi], p] = 1

    col_names = [f'TIRP_{p:04d}_k{t.k}' for p, t in enumerate(tirps)]
    return X, col_names

print('KarmaLego + KLS helpers defined.')


---
## Extension 1 — KLS AUC Evaluation

> **Question:** Do multi-week symbolic patterns mined by KarmaLego outperform the 15-feature baseline in walk-forward prediction?

**Pipeline:**
1. Fit TD4C cutpoints on 2016–2018 training data (5 momentum features → 3 states each)
2. Convert each ticker's weekly values to sorted symbolic intervals
3. Run KarmaLego on 2016–2018 intervals → mine frequent 2-TIRPs (vertical support ≥ 8 tickers)
4. Build a binary KLS feature matrix: for each mined TIRP, flag the 4 weeks immediately following its realisation
5. Walk-forward evaluation with LogReg + XGBoost

**Expected runtime:** ~75 seconds (early-exit optimisation reduces inner loop from O(n²) to O(n×w)).

In [ ]:
t0 = time.time()
primary_kls = load_primary(DATASET_PATH)
print(f'Loaded {len(primary_kls):,} primary rows ({primary_kls["Ticker"].nunique()} tickers)')


In [ ]:
# Fit TD4C cutpoints on 2016-2018 (first training window)
discr_train = primary_kls[primary_kls['Year'] <= 2018]
y_discr     = discr_train[TARGET].values.astype(int)

td4c_cuts: dict[str, list[float]] = {}
for feat in KLS_FEATURES:
    td4c_cuts[feat] = td4c_cutpoints(
        discr_train[feat].values.astype(float), y_discr,
        n_states=3, n_candidates=19
    )

print(f'TD4C fit on {len(discr_train):,} rows (Year <= 2018)')
for feat, cuts in td4c_cuts.items():
    print(f'  {feat:>20s} : {[round(c,4) for c in cuts]}')


In [ ]:
# Build symbolic intervals for the full history (all years)
ticker_intervals_full: dict[str, list[Interval]] = {}
for t, sub in primary_kls.groupby('Ticker'):
    sub = sub.sort_values('WeekIdx')
    ticker_intervals_full[t] = build_ticker_intervals(sub, td4c_cuts, KLS_FEATURES)

# And restricted to the 2016-2018 mining window
mining_primary = primary_kls[primary_kls['Year'] <= 2018]
ticker_intervals_mining: dict[str, list[Interval]] = {}
for t, sub in mining_primary.groupby('Ticker'):
    sub = sub.sort_values('WeekIdx')
    ticker_intervals_mining[t] = build_ticker_intervals(sub, td4c_cuts, KLS_FEATURES)

total_full   = sum(len(v) for v in ticker_intervals_full.values())
total_mining = sum(len(v) for v in ticker_intervals_mining.values())
print(f'Symbolic intervals — full history : {total_full:,}')
print(f'Symbolic intervals — 2016-2018    : {total_mining:,}')


In [ ]:
# Mine TIRPs
print(f'Mining TIRPs  (min_support={MIN_TICKER_SUPPORT}, max_gap={MAX_GAP}, max_k={MAX_K}) ...')
tirps = karmalego_multi(
    ticker_intervals_mining,
    min_ticker_support=MIN_TICKER_SUPPORT,
    max_gap=MAX_GAP,
    max_k=MAX_K,
    verbose=True,
)
print(f'Total TIRPs mined: {len(tirps):,}')


In [ ]:
# Build KLS feature matrix
anchors = primary_kls.reset_index(drop=True)[['Ticker','Date','Year','WeekIdx',TARGET]].copy()
X_kls, kls_cols = build_kls_features(tirps, ticker_intervals_full, anchors, MAX_GAP, MAX_LAG)

# Drop all-zero columns (TIRPs that never fire in any observation window)
active   = X_kls.any(axis=0)
X_kls    = X_kls[:, active]
kls_cols = [c for c, a in zip(kls_cols, active) if a]
density  = X_kls.mean() if X_kls.size > 0 else 0.0
print(f'KLS matrix : {X_kls.shape}  density={density:.4f}  active_features={X_kls.shape[1]:,}')


In [ ]:
y_kls  = primary_kls[TARGET].to_numpy().astype(int)
yr_kls = primary_kls['Year'].to_numpy()

print('--- LogReg ---')
lr_kls = walk_forward(X_kls, y_kls, yr_kls, make_logreg, 'LR-KLS')
print_results(lr_kls, 'LogReg — KLS Features Only')

if _HAS_XGB:
    print('\n--- XGBoost ---')
    xgb_kls = walk_forward(X_kls, y_kls, yr_kls, make_xgb, 'XGB-KLS')
    print_results(xgb_kls, 'XGBoost — KLS Features Only')

print(f'\n[Extension 1 complete in {time.time()-t0:.1f}s]')


---
## Extension 2 — Earnings-Week Flag

> **Question:** Does a simple binary flag for earnings season provide a predictive signal on top of the 15 baseline features?

**Approach:** Flag the 5-week ISO-week windows that fall roughly 3–6 weeks after each quarter end — the typical earnings release window for US tech stocks:

| Quarter | Reports in | ISO weeks |
|---|---|---|
| Q4 (Dec year-end) | January | 1–5, 53 |
| Q1 (Mar) | April | 14–18 |
| Q2 (Jun) | July | 27–31 |
| Q3 (Sep) | October | 40–44 |

This covers **~37%** of all trading weeks.  A raw GapUp rate delta of **+5.2%** (55% vs 50%) suggests genuine seasonality — the walk-forward test reveals whether it survives as a model feature.

**This extension is self-contained** — it loads its own copy of `dataset_clean.csv` and does not depend on Extension 1.

In [ ]:
t0 = time.time()
primary_e2 = load_primary(DATASET_PATH)

# Build earnings-week flag
earnings_weeks = (
    set(range(14, 19)) | set(range(27, 32)) |
    set(range(40, 45)) | set(range(1, 6))   | {53}
)
primary_e2['is_earnings_week'] = primary_e2['WeekOfYear'].isin(earnings_weeks).astype(int)

flag_rate   = primary_e2['is_earnings_week'].mean()
ew_rate     = primary_e2.loc[primary_e2['is_earnings_week'] == 1, TARGET].mean()
non_ew_rate = primary_e2.loc[primary_e2['is_earnings_week'] == 0, TARGET].mean()

print('Earnings-week flag summary')
print(f'  Flagged rows         : {primary_e2["is_earnings_week"].sum():,}  ({flag_rate*100:.1f}%)')
print(f'  GapUp rate (earnings): {ew_rate:.4f}')
print(f'  GapUp rate (other)   : {non_ew_rate:.4f}')
print(f'  Delta                : {ew_rate - non_ew_rate:+.4f}')


In [ ]:
# Visualise GapUp rate by earnings-week status
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: bar chart of GapUp rate
ax = axes[0]
labels = ['Non-earnings\nweeks', 'Earnings\nweeks']
rates  = [non_ew_rate, ew_rate]
bars   = ax.bar(labels, rates, color=['#4c8af5', '#f55c5c'], width=0.5, edgecolor='white')
ax.axhline(primary_e2[TARGET].mean(), color='black', linestyle='--', linewidth=1, label='Overall mean')
ax.set_ylabel('GapUp Rate')
ax.set_title('GapUp Rate by Earnings Season')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=1))
ax.set_ylim(0.44, 0.60)
ax.legend(fontsize=9)
for bar, r in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, r + 0.002, f'{r:.1%}', ha='center', fontsize=10)

# Right: weekly GapUp rate coloured by earnings flag
ax2 = axes[1]
wk_rate = primary_e2.groupby('WeekOfYear')[TARGET].mean()
wk_flag = primary_e2.groupby('WeekOfYear')['is_earnings_week'].first()
colors  = ['#f55c5c' if wk_flag.get(w, 0) else '#4c8af5' for w in wk_rate.index]
ax2.bar(wk_rate.index, wk_rate.values, color=colors, width=1.0)
ax2.axhline(primary_e2[TARGET].mean(), color='black', linestyle='--', linewidth=1)
ax2.set_xlabel('ISO Week of Year')
ax2.set_ylabel('GapUp Rate')
ax2.set_title('GapUp Rate by Week  (red = earnings window)')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))

plt.tight_layout()
plt.show()


In [ ]:
y_e2  = primary_e2[TARGET].to_numpy().astype(int)
yr_e2 = primary_e2['Year'].to_numpy()

X_base_e2     = primary_e2[BASELINE_FEATURES].to_numpy(dtype=float)
X_earnings    = primary_e2[['is_earnings_week']].to_numpy(dtype=float)
X_base_earn   = primary_e2[BASELINE_FEATURES + ['is_earnings_week']].to_numpy(dtype=float)

print('--- LogReg ---')
lr_base_e2   = walk_forward(X_base_e2,   y_e2, yr_e2, make_logreg, 'LR-Baseline')
lr_earn_only = walk_forward(X_earnings,  y_e2, yr_e2, make_logreg, 'LR-EarningsOnly')
lr_base_earn = walk_forward(X_base_earn, y_e2, yr_e2, make_logreg, 'LR-Base+Earnings')

print_results(lr_base_e2,   'LogReg — Baseline (15 features)')
print_results(lr_earn_only, 'LogReg — Earnings Flag Only')
print_results(lr_base_earn, 'LogReg — Baseline + Earnings Flag (16 features)')

nc    = ~lr_base_e2['COVID'].values
delta = lr_base_earn['AUC'].values - lr_base_e2['AUC'].values
print('\n  Per-fold delta  (Base+Earnings) − Baseline:')
for i, fold in enumerate(FOLDS):
    tag = '  <- COVID excluded from mean' if fold['test_year'] == 2020 else ''
    print(f'    {fold["test_year"]}: {delta[i]:+.4f}{tag}')
print(f'  Mean delta (non-COVID): {delta[nc].mean():+.4f}')


In [ ]:
if _HAS_XGB:
    print('--- XGBoost ---')
    xgb_base_e2  = walk_forward(X_base_e2,   y_e2, yr_e2, make_xgb, 'XGB-Baseline')
    xgb_base_earn = walk_forward(X_base_earn, y_e2, yr_e2, make_xgb, 'XGB-Base+Earnings')
    print_results(xgb_base_e2,   'XGBoost — Baseline')
    print_results(xgb_base_earn, 'XGBoost — Baseline + Earnings Flag')
    nc_x    = ~xgb_base_e2['COVID'].values
    delta_x = xgb_base_earn['AUC'].values - xgb_base_e2['AUC'].values
    print(f'\n  XGB mean delta (non-COVID): {delta_x[nc_x].mean():+.4f}')

print(f'\n[Extension 2 complete in {time.time()-t0:.1f}s]')


---
## Extension 3 — Cyclic Temporal + Baseline Combination

> **Question:** Do the 7 engineered cyclic calendar features add discriminative power on top of the 15 baseline features?

The temporal features are sin/cos encodings of calendar cycles (avoiding the discontinuity in raw integers) plus a linear trend:

| Feature | Encoding | Period |
|---|---|---|
| `Month_sin`, `Month_cos` | sin/cos | 12 months |
| `WeekOfYear_sin`, `WeekOfYear_cos` | sin/cos | 52 weeks |
| `Quarter_sin`, `Quarter_cos` | sin/cos | 4 quarters |
| `DaysSinceStart` | linear | — |

Three feature sets are evaluated: **Baseline (15)**, **Temporal only (7)**, **Combined (22)**.

**This extension is self-contained** — it loads `dataset_clean_temporal.csv` directly.

In [ ]:
t0 = time.time()
primary_e3 = load_primary(TEMPORAL_PATH)
print(f'Loaded {len(primary_e3):,} rows from dataset_clean_temporal.csv')
print(f'Temporal columns present: {[c for c in TEMPORAL_FEATURES if c in primary_e3.columns]}')


In [ ]:
y_e3  = primary_e3[TARGET].to_numpy().astype(int)
yr_e3 = primary_e3['Year'].to_numpy()

X_base_e3 = primary_e3[BASELINE_FEATURES].to_numpy(dtype=float)
X_temp    = primary_e3[TEMPORAL_FEATURES].to_numpy(dtype=float)
X_comb    = primary_e3[BASELINE_FEATURES + TEMPORAL_FEATURES].to_numpy(dtype=float)

print(f'Baseline shape : {X_base_e3.shape}')
print(f'Temporal shape : {X_temp.shape}')
print(f'Combined shape : {X_comb.shape}')


In [ ]:
print('--- LogReg ---')
lr_base_e3 = walk_forward(X_base_e3, y_e3, yr_e3, make_logreg, 'LR-Baseline')
lr_temp    = walk_forward(X_temp,    y_e3, yr_e3, make_logreg, 'LR-Temporal')
lr_comb    = walk_forward(X_comb,    y_e3, yr_e3, make_logreg, 'LR-Combined')

print_results(lr_base_e3, 'LogReg — Baseline (15 features)')
print_results(lr_temp,    'LogReg — Temporal Calendar (7 features)')
print_results(lr_comb,    'LogReg — Baseline + Temporal (22 features)')


In [ ]:
# Per-fold comparison table
nc = ~lr_base_e3['COVID'].values
print('\n  Per-fold AUC comparison (LogReg):')
print(f'  {"Year":>6} | {"Baseline":>10} | {"Temporal":>10} | {"Combined":>10} | {"Δ Comb−Base":>12}')
print('  ' + '─'*60)
for i, fold in enumerate(FOLDS):
    covid_tag = ' *' if fold['test_year'] == 2020 else '  '
    b  = lr_base_e3['AUC'].iloc[i]
    t_ = lr_temp['AUC'].iloc[i]
    c  = lr_comb['AUC'].iloc[i]
    print(f'  {fold["test_year"]:>6}{covid_tag}| {b:>10.4f} | {t_:>10.4f} | {c:>10.4f} | {c-b:>+12.4f}')

mb = lr_base_e3['AUC'].values[nc].mean()
mt = lr_temp['AUC'].values[nc].mean()
mc = lr_comb['AUC'].values[nc].mean()
print('  ' + '─'*60)
print(f'  {"Mean":>8}  | {mb:>10.4f} | {mt:>10.4f} | {mc:>10.4f} | {mc-mb:>+12.4f}')


In [ ]:
if _HAS_XGB:
    print('--- XGBoost ---')
    xgb_base_e3 = walk_forward(X_base_e3, y_e3, yr_e3, make_xgb, 'XGB-Baseline')
    xgb_temp    = walk_forward(X_temp,    y_e3, yr_e3, make_xgb, 'XGB-Temporal')
    xgb_comb    = walk_forward(X_comb,    y_e3, yr_e3, make_xgb, 'XGB-Combined')
    print_results(xgb_base_e3, 'XGBoost — Baseline')
    print_results(xgb_temp,    'XGBoost — Temporal Calendar')
    print_results(xgb_comb,    'XGBoost — Baseline + Temporal')
    nc_x = ~xgb_base_e3['COVID'].values
    xb = xgb_base_e3['AUC'].values[nc_x].mean()
    xt = xgb_temp['AUC'].values[nc_x].mean()
    xc = xgb_comb['AUC'].values[nc_x].mean()
    print(f'\n  XGB mean AUC (non-COVID):')
    print(f'    Baseline : {xb:.4f}')
    print(f'    Temporal : {xt:.4f}')
    print(f'    Combined : {xc:.4f}  (Δ {xc-xb:+.4f} vs baseline)')

print(f'\n[Extension 3 complete in {time.time()-t0:.1f}s]')


---
## Summary of Findings

| Extension | LR AUC (non-COVID) | XGB AUC (non-COVID) | vs Baseline |
|---|---|---|---|
| Baseline (15 features) | **0.5650** | **0.5759** | — |
| KLS only (Extension 1) | 0.5093 | 0.5284 | ↓ weaker than baseline |
| Earnings flag only (Ext 2) | 0.5109 | — | ↓ weaker alone |
| Baseline + Earnings (Ext 2) | 0.5620 | 0.5760 | ≈ flat (−0.003 / +0.0001) |
| Temporal only (Ext 3) | 0.4866 | 0.4971 | ↓ below chance |
| Baseline + Temporal (Ext 3) | 0.5621 | 0.5391 | ↓ LogReg flat, XGB −0.037 |

### Key takeaways

1. **Baseline features dominate.** All three temporal/pattern approaches fail to provide consistent additive lift above the 15 technical + fundamental indicators.

2. **KLS patterns are marginal.** 897 frequent 2-TIRPs mined from 5 momentum features yield AUC of 0.51–0.53 — above chance but well below baseline. More features or 3-TIRPs may help but come at a steep compute cost.

3. **Earnings flag is neutral.** Despite a raw +5.2% GapUp rate in earnings windows, the flag provides zero additive value once baseline indicators are present — they already capture whatever the earnings calendar signals.

4. **Calendar features hurt XGBoost.** Adding cyclic temporal encodings costs XGB −0.037 AUC — the tree model over-splits on noisy calendar boundaries. The 2020 COVID fold is a notable exception (temporal AUC spikes to 0.65+), confirming that date-driven behaviour was uniquely strong that year.

5. **Recommended next steps:**
   - Try sector-normalised technical features to remove cross-ticker scale differences
   - Experiment with interaction terms (e.g. `MACD × is_earnings_week`)
   - Increase `MAX_K=3` with the early-exit optimisation to see if 3-TIRPs push KLS AUC above 0.55
